# Task 4 — Join, Transformation Rules and Testing

## Objective

This notebook applies transformation rules to the validated Pluralsight dataset produced in Task 3 and tests those transformation rules.

## Transformation Rules

The following transformation rules are applied to the validated Pluralsight dataset:

| Rule | Transformation | Input Column(s) | Output Column | Description |
|------|----------------|-----------------|---------------|-------------|
| **R1** | Calculate Word Count | `content` | `word_count` | Calculate the total number of words in the article content. |
| **R2** | Handle Missing Authors | `author` | `author` | Replace missing or empty author values with `"Unknown"`. |
| **R3** | Identify Long-Form Content | `word_count` | `is_long_form` | Set `is_long_form` to `True` when `word_count > 500`; otherwise, set it to `False`. |

These transformations enrich and standardize the validated dataset before it is used in the final integrated pipeline.

The final output is:

`../data/processed/final.csv`

## Imports and Data Ingestion

In [1]:
import os
import pandas as pd

# ==============================================================================
# INPUT / OUTPUT PATHS
# ==============================================================================
INPUT_VALIDATED_PATH = "../data/interim/validated.csv"
OUTPUT_FINAL_PATH = "../data/processed/final.csv"

# Double check that our input source file exists before executing
if not os.path.exists(INPUT_VALIDATED_PATH):
    raise FileNotFoundError(
        f"Missing required input file from Task 3: {INPUT_VALIDATED_PATH}"
    )

df_validated = pd.read_csv(INPUT_VALIDATED_PATH)
print(f"Loaded validated data successfully. Shape: {df_validated.shape}")

Loaded validated data successfully. Shape: (20, 10)


## Transformation Rules Implementation

In [2]:
# Create a copy to perform safe data mutations
df_transformed = df_validated.copy()

# Rule 1: Calculate word count from the content column
df_transformed["word_count"] = (
    df_transformed["content"].astype(str).apply(lambda x: len(x.split()))
)

# Rule 2: Standardize missing values in optional fields (Impute 'Unknown' for blank authors)
df_transformed["author"] = df_transformed["author"].fillna("Unknown")

# Rule 3: Flag long-form technical content (Content with > 500 words gets marked as True)
df_transformed["is_long_form"] = df_transformed["word_count"] > 500

print("Transformation rules successfully applied to data columns.")

Transformation rules successfully applied to data columns.


## Unit Testing & Assertions Block

In [3]:
print("=========================================================================")
print("RUNNING PIPELINE QUALITY CONTROL TEST ASSERTIONS")
print("=========================================================================")

# Test Case 1: Ensure word count calculations are strictly positive numbers
assert (df_transformed["word_count"] >= 0).all(), (
    "Test Fail: Found negative word counts!"
)
print("Test 1 Passed: All word counts are valid positive integers.")

# Test Case 2: Ensure there are absolutely no true null values left in the author field
assert not df_transformed["author"].isna().any(), (
    "Test Fail: Found unhandled NaN values in author column!"
)
print("Test 2 Passed: Author field imputation verified successfully.")

# Test Case 3: Verify long-form flag logic aligns accurately with word count limits
sample_check = df_transformed[df_transformed["word_count"] <= 500]
assert not sample_check["is_long_form"].any(), (
    "Test Fail: Flagged long form content incorrectly below threshold!"
)
print("Test 3 Passed: Long-form structural boolean flags validated.")

print("\nALL PIPELINE ASSERTION TESTS PASSED SUCCESSFULLY!")


RUNNING PIPELINE QUALITY CONTROL TEST ASSERTIONS
Test 1 Passed: All word counts are valid positive integers.
Test 2 Passed: Author field imputation verified successfully.
Test 3 Passed: Long-form structural boolean flags validated.

ALL PIPELINE ASSERTION TESTS PASSED SUCCESSFULLY!


## Final Checkpoint Target Export

In [4]:
# Ensure the processed folder structure path exists safely
os.makedirs(os.path.dirname(OUTPUT_FINAL_PATH), exist_ok=True)

try:
    # Save the polished, analysis-ready production file to disk
    df_transformed.to_csv(OUTPUT_FINAL_PATH, index=False, encoding="utf-8")
    print(
        f"\nPipeline complete! Final dataset saved cleanly to: {OUTPUT_FINAL_PATH}"
    )

    # Display final row counts and data look for your submission records
    print(f"Final exported row count: {df_transformed.shape}")
    print("\nColumns included in the final delivery file:")
    print(list(df_transformed.columns))

except PermissionError:
    print(
        f"\nERROR: Please close Excel if it is viewing '{OUTPUT_FINAL_PATH}' and re-run this cell."
    )



Pipeline complete! Final dataset saved cleanly to: ../data/processed/final.csv
Final exported row count: (20, 12)

Columns included in the final delivery file:
['source', 'category', 'title', 'author', 'publication_date', 'description', 'tags', 'url', 'content', 'scraped_at', 'word_count', 'is_long_form']
